# Resultados Fase 1 — Estado laboral macro (Ocupado / Parado / Inactivo)

**TFG:** Machine Learning explicable para analizar el estado laboral y la calidad del empleo en España.

Este notebook documenta el modelado **jerárquico anti-leakage** de la Fase 1:

1. **Etapa A:** Activo vs Inactivo (solo demografía) → Random Forest
2. **Etapa B:** Ocupado vs Parado sobre activos (solo demografía) → LightGBM
3. **Combinado A→B** con umbrales F1 calibrados (3 clases)

Figuras en `reports/figures/fase1/`. Métricas en `reports/fase1/comparacion_modelos.json` y `reports/memoria/metricas_maestras.json`.

**Mensaje clave:** la demografía explica bien la **participación** (PR-AUC ≈ 0,94), pero no separa **paro vs ocupación** (PR-AUC ≈ 0,27).


## 1. Métricas y comparación de modelos

Carga los artefactos generados por `python run_model_fase1.py` (reentrenamiento con búsqueda ampliada, baseline y umbrales F1).


In [ ]:
import json
from pathlib import Path

from IPython.display import Markdown, display

ROOT = Path.cwd().resolve()
if ROOT.name == "modelado":
    ROOT = ROOT.parents[1]
elif ROOT.name == "notebooks":
    ROOT = ROOT.parent

with open(ROOT / "reports/fase1/comparacion_modelos.json", encoding="utf-8") as f:
    comp = json.load(f)
with open(ROOT / "reports/memoria/metricas_maestras.json", encoding="utf-8") as f:
    maestras = json.load(f)

cfg = comp["config_busqueda"]
a, b = comp["etapa_a"], comp["etapa_b"]
comb = maestras["fase1"]["combinado_umbrales_calibrados"]

print("=== CONFIGURACIÓN ===")
print(f"Búsqueda HP: n_iter={cfg['n_iter']}, muestra={cfg['tamano_muestra']:,}, CV={cfg['cv_folds']}")
print(f"Split temporal: train ≤ {maestras['corte_temporal']['train_hasta']} · test ≥ {maestras['corte_temporal']['test_desde']}")
print()

for etapa, key in [("A — Activo vs Inactivo", "etapa_a"), ("B — Ocupado vs Parado", "etapa_b")]:
    e = comp[key]
    gan = e["ganador"]
    m = e["comparacion"][gan]
    bl = e["comparacion"]["baseline_prevalencia"]
    opt = e["metricas_test_umbral_optimo"]
    print(f"=== Etapa {etapa} ===")
    print(f"Ganador: {gan}")
    print(f"PR-AUC test: {m['pr_auc']:.4f}  |  ROC-AUC: {m['roc_auc']:.4f}  |  F1 @0.50: {m['f1']:.4f}")
    print(f"Baseline PR-AUC: {bl['pr_auc']:.4f}")
    print(f"Umbral F1 (train): {e['umbral_optimo_f1_train']:.3f}  →  F1 test: {opt['f1']:.4f}")
    print()

print("=== Combinado A→B (umbrales calibrados) ===")
print(f"Accuracy: {comb['accuracy']:.2f}")
print(f"F1 Inactivo: {comb['f1_inactivo']:.2f}  |  F1 Ocupado: {comb['f1_ocupado']:.2f}  |  F1 Parado: {comb['f1_parado']:.2f}")
print(f"Macro-F1 aprox.: {comb['macro_f1_aprox']:.2f}")

informe = ROOT / "reports/fase1/informe_final.txt"
if informe.exists():
    display(Markdown("### Informe de clasificación (test)\n\n```\n" + informe.read_text(encoding="utf-8") + "\n```"))


## 2. Comparativa rápida de candidatos (PR-AUC test)

Tabla sintética para la memoria: los tres algoritmos en cada etapa.


In [ ]:
import pandas as pd

filas = []
for etapa, key in [("A", "etapa_a"), ("B", "etapa_b")]:
    for modelo, m in comp[key]["comparacion"].items():
        if modelo == "baseline_prevalencia":
            continue
        if "pr_auc" not in m:
            continue
        filas.append({
            "Etapa": etapa,
            "Modelo": modelo.replace("_", " ").title(),
            "PR-AUC": round(m["pr_auc"], 4),
            "ROC-AUC": round(m.get("roc_auc", float("nan")), 4),
            "F1 @0.50": round(m.get("f1", float("nan")), 4),
            "Ganador": "✓" if comp[key]["ganador"] == modelo else "",
        })

tabla = pd.DataFrame(filas).sort_values(["Etapa", "PR-AUC"], ascending=[True, False])
display(tabla.style.hide(axis="index"))


## 3. Galería de figuras


In [ ]:
from IPython.display import Image, Markdown, display


def _mostrar_fig(path):
    if not path.exists():
        display(Markdown(f"*No encontrada:* `{path.relative_to(ROOT)}`"))
        return
    display(Markdown(f"#### `{path.name}`"))
    display(Image(data=path.read_bytes(), width=720))


# reports/figures/fase1

_mostrar_fig(ROOT / "reports/figures/fase1" / "etapa_a__activo_vs_inactivo_curva_pr.png")

_mostrar_fig(ROOT / "reports/figures/fase1" / "etapa_a__activo_vs_inactivo_matriz_confusion.png")

_mostrar_fig(ROOT / "reports/figures/fase1" / "etapa_a__activo_vs_inactivo_matriz_confusion_umbral_optimo.png")

_mostrar_fig(ROOT / "reports/figures/fase1" / "etapa_a__activo_vs_inactivo_shap_summary.png")

_mostrar_fig(ROOT / "reports/figures/fase1" / "etapa_a__activo_vs_inactivo_shap_importancia_barras.png")

_mostrar_fig(ROOT / "reports/figures/fase1" / "etapa_b__ocupado_vs_parado_curva_pr.png")

_mostrar_fig(ROOT / "reports/figures/fase1" / "etapa_b__ocupado_vs_parado_matriz_confusion.png")

_mostrar_fig(ROOT / "reports/figures/fase1" / "etapa_b__ocupado_vs_parado_matriz_confusion_umbral_optimo.png")

_mostrar_fig(ROOT / "reports/figures/fase1" / "etapa_b__ocupado_vs_parado_shap_summary.png")

_mostrar_fig(ROOT / "reports/figures/fase1" / "etapa_b__ocupado_vs_parado_shap_importancia_barras.png")

_mostrar_fig(ROOT / "reports/figures/fase1" / "combinado_matriz_confusion.png")


## 4. Decisiones que ilustra la Fase 1

| Decisión | Evidencia |
|---|---|
| Solo demografía en Fase 1 (anti-leakage) | Variables laborales excluidas; target macro no filtra al predictor |
| PR-AUC como métrica principal | Clases desbalanceadas (parado ≈ 9 %) |
| Umbrales F1 calibrados (A=0,275 · B=0,625) | Matrices @0,50 vs umbral óptimo |
| Etapa B no mejora mucho con más HP | LogReg, RF y LightGBM cercanos → techo informativo |
| Cascada A→B | Accuracy combinada ≈ 0,72; paro es la clase más difícil (F1 ≈ 0,25) |

Con esto la **Fase 1** queda documentada para memoria y defensa.
